[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/https://github.com/maurergroup/MLinCP/Module_XI_Generative_Models/WS9_Generative_Models/WS5_Generative_Models.ipynb)

# WS 10 — Machine-learning Methods for Rare Events Sampling

**Machine Learning in Computational Physics** · University of Vienna

---

<div class="alert alert-block alert-danger">

**Assignment submission**

When you submit this notebook for assignment, make sure that all tasks below (blue boxes) are completed (with code) and all questions are answered (with extra markdown responses). Make sure that the notebook runs correctly when all cells are executed in the right order from top to bottom. Before submission, do not clear outputs. Leave all the outputs from the last run included.
</div>

## 0. Prelude

In this workshop, we will investigate how machine learning can be used to approximate the committor function, one of the central quantities in the study of rare events and activated processes. The committor of a configuration is defined as the probability that a trajectory initiated from that configuration reaches the product state before returning to the reactant state. As such, it provides an ideal reaction coordinate and offers valuable insight into the mechanism of a transition.

To estimate the committor, we will combine Transition Path Sampling (TPS) simulations with a neural network model. Our goal is to implement a simplified version of the AIMMD (Artificial Intelligence for Molecular Mechanism Discovery) framework introduced in [Jung2023]. Rather than considering a complex molecular system, we will focus on the two-dimensional Wolfe–Quapp potential, which retains the essential features of a rare-event problem while remaining computationally inexpensive. Starting from an ensemble of TPS shooting trajectories, we will train a neural network to learn the committor function and use the resulting model to identify the transition-state region separating the two metastable states. Although we will perform only a single iteration of the AIMMD procedure, we will see that this is already sufficient to obtain an accurate approximation of the committor and to illustrate the key ideas behind self-consistent machine-learning approaches to rare-event sampling.


The workshop relies exclusively on widely used Python libraries. Numerical computations are performed using NumPy, while PyTorch is employed to define, train, and evaluate the neural-network model used to approximate the committor function. We also use Joblib to parallelize computationally intensive tasks, such as the estimation of committor probabilities from large ensembles of trajectories. Visualization of the potential-energy surface, trajectories, and learned committor is carried out using Matplotlib. The following imports provide all the functionality required throughout the workshop.

In [ ]:
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm

from functools import partial

import matplotlib.pyplot as plt
from matplotlib.patches import Circle

import pathlib
# Locate repo root (contains pyproject.toml) — works in Docker, VS Code, and local Jupyter
_p = pathlib.Path().resolve()
while not (_p / 'pyproject.toml').exists() and _p != _p.parent:
    _p = _p.parent
DATA_DIR = _p.joinpath('data')

## 1. Potential Energy Surface and Stable States

### 1A. Wolfe-Quapp potential

To illustrate the AIMMD workflow, we will consider the Wolfe–Quapp potential, a two-dimensional model that has become a standard benchmark for studying rare events, transition pathways, and reaction-coordinate discovery. Despite its simplicity, the potential captures many of the key features encountered in molecular systems: the presence of multiple metastable states, a free-energy barrier separating them, and a non-trivial transition region through which reactive trajectories must pass.

The Wolfe–Quapp potential is defined as a fourth-order polynomial in the two Cartesian coordinates $x$ and $y$. The resulting energy landscape contains two stable minima, denoted as states A and B, separated by a saddle region corresponding to the transition state. 

$$
V(x,y)
=
s \Big(
0.969233\,x^4
-0.480614\,x^3y
+0.184602\,x^2y^2
-2.155285\,x^2
+0.480614\,xy^3
-1.46487\,xy
-0.285146\,x
+0.969233\,y^4
-3.84471\,y^2
+0.136719\,y
\Big)
+
V_b
$$

with

$$
s=\frac{V_b}{6.76245}
$$

A particle undergoing stochastic dynamics on this surface spends most of its time fluctuating within one of the minima and only rarely crosses the barrier to reach the other state. These rare transitions are precisely the events we aim to characterize through the committor function.

An important advantage of the Wolfe–Quapp potential is that the complete potential-energy surface can be visualized directly. This allows us to compare the learned committor with the underlying landscape and to inspect how the transition-state region emerges from the machine-learning procedure. Throughout the workshop, the barrier height will be specified in units of $k_B T$, making it straightforward to control the rarity of the transition events.

The `WolfeQuapp` class provides a convenient implementation of the potential energy surface and the corresponding force field. The class stores the polynomial coefficients defining the potential and the coordinates of the two minima, which will later be used to define the stable states. The constructor accepts a barrier height and rescales the potential accordingly. Two main methods are provided:

* `energy(coords)`: evaluates the potential energy $V(x,y)$ at one or more configurations. The input can be either a single configuration of shape `(2,)` or an array of configurations with shape `(..., 2)`.
* `force(coords)`: evaluates the force $\mathbf{F}=-\nabla V$, which is required to propagate overdamped Langevin dynamics. As for the energy function, the method is fully vectorized and can operate on batches of configurations.

The force is obtained analytically by differentiating the polynomial expression of the potential. The resulting class therefore provides all the ingredients needed to perform stochastic dynamics, generate transition paths, and ultimately construct a machine-learning approximation of the committor function.


In [ ]:
class WolfeQuapp:
    """
    Wolfe-Quapp potential.

    Barrier height is specified in units of k_B T.
    """

    # Polynomial coefficients
    X4 = 0.969233
    X3Y = -0.480614
    X2Y2 = 0.184602
    X2 = -2.155285
    XY3 = 0.480614
    XY = -1.46487
    X = -0.285146

    Y4 = 0.969233
    Y2 = -3.84471
    Y = 0.136719

    # Minima
    POS_STATE_A = np.array([-1.30096, -1.33310], dtype=np.float64)
    POS_STATE_B = np.array([ 1.34950,  1.31873], dtype=np.float64)

    def __init__(self, barrier_height: float = 7.0):

        if barrier_height <= 0:
            raise ValueError("barrier_height must be positive")

        self.barrier_height = barrier_height

        # Reference barrier ≈ 6.76245 kBT
        self.scale = barrier_height / 6.76245

    def energy(self, coords: np.ndarray) -> np.ndarray:
        """
        Parameters
        ----------
        coords : (..., 2)

        Returns
        -------
        (...,)
        """

        coords = np.asarray(coords)

        x = coords[..., 0]
        y = coords[..., 1]

        V = (
            self.X4 * x**4
            + self.X3Y * x**3 * y
            + self.X2Y2 * x**2 * y**2
            + self.X2 * x**2
            + self.XY3 * x * y**3
            + self.XY * x * y
            + self.X * x
            + self.Y4 * y**4
            + self.Y2 * y**2
            + self.Y * y
        )

        return self.scale * V + self.barrier_height

    def force(self, coords: np.ndarray) -> np.ndarray:
        """
        Force = -∇V

        Parameters
        ----------
        coords : (..., 2)

        Returns
        -------
        (..., 2)
        """

        coords = np.asarray(coords)

        x = coords[..., 0]
        y = coords[..., 1]

        Fx = -self.scale * (
            4 * self.X4 * x**3
            + 3 * self.X3Y * x**2 * y
            + 2 * self.X2Y2 * x * y**2
            + 2 * self.X2 * x
            + self.XY3 * y**3
            + self.XY * y
            + self.X
        )

        Fy = -self.scale * (
            self.X3Y * x**3
            + 2 * self.X2Y2 * x**2 * y
            + 3 * self.XY3 * x * y**2
            + self.XY * x
            + 4 * self.Y4 * y**3
            + 2 * self.Y2 * y
            + self.Y
        )

        return np.stack((Fx, Fy), axis=-1)

### 1B. Circular stable states

To study transitions on the Wolfe–Quapp potential, we must first define the metastable states between which the system evolves. In molecular simulations, stable states are often characterized through a set of collective variables (CVs), low-dimensional descriptors that can partition the configuration space in disjoint areas. Rather than working directly with the full configuration space, one defines regions in terms of these collective variables and classifies configurations according to whether they belong to a given state.

In this workshop, the configuration space is already two-dimensional and therefore easy to visualize. Nevertheless, we introduce a simple collective variable framework to mimic the procedures used in realistic molecular systems. Our collective variable is based on the scaled distance from a reference point. Given a configuration $\mathbf{x}=(x,y)$ and a state center $\mathbf{x}_0=(x_0,y_0)$, we first translate the coordinates so that the state center becomes the origin,

$$
x' = x - x_0,
\qquad
y' = y - y_0,
$$

and optionally rotate the coordinate system by an angle $\theta$. The resulting coordinates are then scaled along two principal axes of lengths $a$ and $b$. The collective variable is defined as

$$
\zeta(\mathbf{x})
=
\left(\frac{x_{\mathrm{rot}}}{a}\right)^2
+
\left(\frac{y_{\mathrm{rot}}}{b}\right)^2 .
$$

For the special case $a=b$, this corresponds to the squared distance from the state center measured in units of the chosen radius. More generally, the definition allows elliptical state boundaries with arbitrary orientation.

The class `circular_cv` implements this collective variable. Given one or more configurations, it evaluates the quantity $\zeta$, which measures how far a configuration lies from a chosen reference point. Because the implementation is fully vectorized, it can be applied efficiently to individual configurations or large batches of configurations.

In [ ]:
class circular_cv:

    def __init__(self, axes: tuple = (1.0, 1.0), angle: float = 0.0):

        self.a, self.b = axes

        self.cos_t = np.cos(angle)
        self.sin_t = np.sin(angle)

    def compute(self, current_x : np.ndarray, center: tuple = (0.0, 0.0)) -> float:
        """
        Calculates the collective variable zeta given a configuration current_x.

        Parameters
        ----------
        current_x : np.ndarray
            Current configuration. The shape of the array(current_x.shape) can vary depending on the system which is simulated.

        Returns
        -------
        zeta : float
            CV corresponding the provided configuration.
        """

        x_shifted = current_x[..., 0] - center[0]
        y_shifted = current_x[..., 1] - center[1]

        # Rotate coordinates
        x_rot = self.cos_t * x_shifted + self.sin_t * y_shifted
        y_rot = -self.sin_t * x_shifted + self.cos_t * y_shifted

        # Elliptical scaled distance
        zeta = (x_rot / self.a)**2 + (y_rot / self.b)**2

        return zeta

The class `circular_state` uses this collective variable to define a stable state. A configuration is considered to belong to the state whenever

$$
\zeta(\mathbf{x}) < r^2,
$$

where (r) is the state radius. In other words, the state corresponds to the interior of a circle (or, more generally, an ellipse) centered at a specified location in configuration space. The method `is_in_state` evaluates this condition and returns `True` if the configuration belongs to the state and `False` otherwise.

For the Wolfe–Quapp potential, we will place two such states around the minima of the energy landscape. These state definitions will be used throughout the workshop to determine whether a trajectory has reached state A or state B, to classify transition paths, and ultimately to estimate the committor function.

In [ ]:
class circular_state:

    def __init__(self, center : tuple = (0, 0), axes :  tuple = (1, 1), angle : float = 0, radius : float = 1.):

        self.center = center
        self.radius = radius
        self.cv = circular_cv(axes=axes, angle=angle)

    def is_in_state(self, current_x : np.ndarray) -> bool:
    
        """
        Returns a bool or bool array which is true if current_x is within in the bounds.

        Parameters
        ----------
        current_x : np.ndarray
            Current configuration. The shape of the array(current_x.shape) can vary depending on the system which is simulated.

        cv_function : callable
            Function that accepts the current configuration and maps it onto the collective variable.
            
        bounds : (float, float)
            The lower and upper bounds for x to be considered within the stable state


        Returns
        -------
        inState : bool
            True for each x in current_x if x is within the bounds
        """
        cv_values = self.cv.compute(current_x, self.center)
        
        return cv_values < self.radius**2

Before generating trajectories, it is useful to visualize the energy landscape and verify that the stable states have been defined correctly. In the following cell, we evaluate the Wolfe–Quapp potential on a regular grid covering the region of interest and display the resulting potential-energy surface using contour plots.

The figure combines two complementary visualizations. A filled contour plot (`contourf`) is used to represent the value of the potential energy through color, providing an intuitive picture of the two metastable basins and the barrier separating them. In addition, contour lines (`contour`) are drawn to emphasize the structure of the energy landscape and make it easier to identify minima, saddle regions, and transition pathways.

The two stable states are then superimposed on the potential-energy surface. Using the centers corresponding to the minima of the Wolfe–Quapp potential, we draw circular regions representing states A and B. These circles correspond to the regions in configuration space for which the state indicator function returns `True`. Their position and size therefore provide a direct visual confirmation of the state definitions introduced in the previous section.

The resulting figure serves two important purposes. First, it allows us to inspect the topology of the potential energy surface and identify the transition region separating the two metastable basins. Second, it provides a convenient reference that we will reuse throughout the workshop to visualize trajectories, shooting points, learned committor contours, and transition-state configurations. Because the entire system is two-dimensional, we can directly compare the machine-learning results with the underlying energy landscape, an opportunity that is rarely available in realistic molecular simulations.


In [ ]:
# Instantiate potential
wq = WolfeQuapp(barrier_height=7.0)

state_A = circular_state(
    center=tuple(wq.POS_STATE_A),
    radius=0.5
)

state_B = circular_state(
    center=tuple(wq.POS_STATE_B),
    radius=0.5
)

states = [state_A, state_B]

# Grid
x = np.linspace(-2.25, 2.25, 400)
y = np.linspace(-2.25, 2.25, 400)

X, Y = np.meshgrid(x, y)

coords = np.stack(
    [X.ravel(), Y.ravel()],
    axis=1
)

# Evaluate energy
V = wq.energy(coords)

V = V.reshape(X.shape)

fig, ax = plt.subplots(figsize=(8, 6))

levels = np.linspace(0, 20, 21)

contours = ax.contour(
    X,
    Y,
    V,
    levels=levels,
    colors="0.5",
    linewidths=1
)

filled = ax.contourf(
    X,
    Y,
    V,
    levels=levels,
    cmap="viridis",
    extend="max"
)

plt.colorbar(
    filled,
    ax=ax,
    label=r"$V(x,y)$ [$k_B T$]"
)

# ----- Plot states -----

A = np.asarray(state_A.center)
B = np.asarray(state_B.center)

# ----- Plot states -----

for label, state in zip(["A", "B"], [state_A, state_B]):

    circle = Circle(
        state.center,
        radius=state.radius,
        facecolor=(1, 1, 1, 0.5),  # white, alpha=0.5
        edgecolor=(0, 0, 0, 1.0),  # black, alpha=1.0
        linewidth=2.5,
        zorder=20,
    )

    ax.add_patch(circle)

    ax.text(
        state.center[0],
        state.center[1],
        label,
        ha="center",
        va="center",
        fontsize=14,
        fontweight="bold",
        color="black",
        zorder=21,
    )

# -----------------------

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Wolfe–Quapp Potential")

ax.set_aspect("equal")
# ax.legend()

plt.tight_layout()
plt.show()

## 2. Dynamics

### 2A. Overdamped Langevin Dynamics via Euler-Maruyama integration

To generate trajectories on the Wolfe–Quapp potential, we model the dynamics of a particle undergoing overdamped Langevin motion. In this regime, inertial effects are neglected and the evolution of the system results from the competition between deterministic forces arising from the potential energy surface and random thermal fluctuations originating from the surrounding environment.

The overdamped Langevin equation can be written as

$$
d\mathbf{x}
=
D\beta \mathbf{F}(\mathbf{x})\,dt
+
\sqrt{2D}\,d\mathbf{W},
$$

where $\mathbf{x}$ denotes the configuration of the system, $\mathbf{F}=-\nabla V$ is the force derived from the potential energy surface, $D$ is the diffusion coefficient, $\beta=(k_B T)^{-1}$ is the inverse temperature, and $d\mathbf{W}$ represents a Wiener process describing thermal noise.

To integrate this stochastic differential equation numerically, we employ the Euler–Maruyama scheme. Given a configuration $\mathbf{x}_i$ at time step $i$, the next configuration is obtained according to

$$
\mathbf{x}_{i+1}
=
\mathbf{x}_i
+
D\beta\,\mathbf{F}(\mathbf{x}_i)\,\Delta t
+
\sqrt{2D\Delta t}\,\mathbf{g}_i,
$$

where $\Delta t$ is the integration time step and $\mathbf{g}_i$ is a vector of independent random numbers drawn from a standard normal distribution.

The update consists of two contributions. The first term,

$$
D\beta\,\mathbf{F}(\mathbf{x}_i)\,\Delta t,
$$

is a deterministic drift that drives the system downhill on the potential-energy surface. The second term,

$$
\sqrt{2D\Delta t}\,\mathbf{g}_i,
$$

introduces stochastic fluctuations that mimic the effect of thermal collisions with the environment. These random perturbations allow the system to occasionally overcome the free-energy barrier separating the two metastable states and generate the rare transitions that are the focus of this workshop.

The function `update_positions` implements a single Euler–Maruyama integration step. Given the current configuration, the corresponding force, the inverse temperature $\beta$, the integration time step $\Delta t$, and the diffusion coefficient $D$, it generates a random Gaussian displacement, computes the deterministic and stochastic contributions to the motion, and returns the updated configuration. By repeatedly applying this propagator, we can construct trajectories that explore the Wolfe–Quapp potential and eventually transition between states A and B.


In [ ]:
def update_positions(current_x : np.ndarray, force : np.ndarray, beta : float, dt : float, diffusion_coeff : float) -> np.ndarray:
    """
    Update the positions using overdamped langevin dynamics:

        x_(i+1) = x_i + D * beta * force * dt + sqrt(2 * D * dt) * g

    where x_i are the positions at i (current_x) and x_(i+1) are the updated positions (next_x). 
    The diffusion coefficient D (diffusion_coeff), the timestep dt, the temperature in form of 
    beta and the force are needed for the propagation. The factor g is a random number from a 
    standard normal distribution.

    Parameters
    ----------
    current_x : np.ndarray
        Current configuration to be propagated. The shape of the array(current_x.shape) can vary depending on the system which is simulated.
        
    force : np.ndarray
        The force corresponding to current_x. This has to be of the same shape as current_x. 

    beta : float
        Beta determines the simulation temperature, it is equivalent to 1/kT. Must be greater than 0. 

    dt : float
        The simulation timestep for the propagation of current_x. Must be greater than 0. Decrease this or diffusion_coeff if you experience an unstable configuration.
    
    diffusion_coeff : float
        The diffusion coefficient for the propagation of current_x determining the magnitude of random "bumps". Must be greater than 0. Decrease this or diffusion_coeff if you experience an unstable configuration.
        
    Returns
    -------
    new_x : np.ndarray
        Updated Configuration.

    """

    assert dt > 0, "Timestep must be positive."
    assert diffusion_coeff > 0, "Diffusion coefficient must be positive."
    assert beta > 0, "Temperature must be positive."
    assert current_x.shape == force.shape, "Force and position vector must be of the same size, check your force function."

    # Draw random number from a standard normal distribution
    gauss_rand = np.random.randn(*current_x.shape).astype(np.float32)
    
    # Calculate displacement
    prefactor = np.sqrt(2 * diffusion_coeff * dt)
    dx = diffusion_coeff * dt * force * beta + prefactor * gauss_rand

    # Update the configuration x
    next_x = current_x + dx
    
    return next_x

### 2B. Generating Trajectories

Having defined a one-step propagator for the overdamped Langevin dynamics, we can now generate complete trajectories. In the context of Transition Path Sampling, a trajectory is initiated from a chosen configuration, called the *shooting point*, and propagated forward in time until it reaches one of the predefined stable states.

The function `generate_path` implements this procedure. Starting from a shooting point $\mathbf{x}_0$, the configuration is evolved according to the Euler–Maruyama integrator introduced above. At every integration step, the force is evaluated, a new configuration is generated, and the resulting configuration is checked against the state definitions. The propagation continues until one of two conditions is met:

1. The trajectory reaches one of the stable states.
2. A predefined maximum number of integration steps is exceeded.

The first condition corresponds to the physically relevant stopping criterion. Once a trajectory enters either state A or state B, its subsequent evolution no longer provides useful information about the transition process. The second condition serves as a safety mechanism that prevents trajectories from running indefinitely in the unlikely event that neither state is reached. This is not really relevant in this scenario but it can be relevant in other forms of potential energies.

To reduce memory consumption, configurations are not necessarily stored at every integration step. Instead, the parameter `configuration_output_frequency` controls how frequently configurations are recorded. For example, a value of 10 stores one configuration every ten integration steps. This strategy is commonly used in molecular simulations, where trajectories may contain millions of integration steps but only a subset of configurations is needed for subsequent analysis.

From an algorithmic perspective, the procedure can be summarized as follows:

1. Initialize the trajectory with the shooting point.
2. Check whether the shooting point already belongs to a stable state.
3. If not, repeatedly:

   * evaluate the force,
   * perform one Euler–Maruyama integration step,
   * store the configuration if the output frequency criterion is satisfied,
   * check whether a stable state has been reached.
4. Stop when a state is reached or the maximum path length is exceeded.

The output of the function is a NumPy array containing the recorded configurations along the trajectory. These trajectories form the basic data used throughout the remainder of the workshop. In particular, they will be used to construct transition paths, generate training data for the neural network committor model, and estimate the probability that a configuration reaches state B before state A.


In [ ]:
def generate_path(shooting_point : np.ndarray, 
                  force_function : callable,
                  max_path_length : int,
                  states : list,
                  configuration_output_frequency : int,
                  beta : float, 
                  timestep : float, 
                  diffusion_coefficient : float):
    """
    Returns a trajectory started from the shooting point and that ends when a state is reached.

    Parameters
    ----------
    force_function: callable
        Function that computes the force for the specific system simulated

    shooting_point : np.ndarray
        Initial configuration of the trajectory.

    max_path_length : int
        MAXIMUM number of integration steps of the trajectory
        
    states : list
        List of state indicator functions to check against if current_x is in a state (for use [state_A_indicator, state_B_indicator])
        
    configuration_output_frequency : int
        Output frequency
        
    beta : float
        1 / kT

    timestep : float
        Timestep of integration

    diffusion_coefficient : float
        Diffusion coefficient of the overdamped dynamics

    Returns
    -------
    trajectory : np.ndarray
        Trajectory started from shooting_point as initial configuration
    """

    previous_x = shooting_point.copy()

    trajectory = [previous_x]
    state_reached = False

    for state in states:
        if state.is_in_state(previous_x): 
            state_reached = True

    if not state_reached:
        for step in range(max_path_length):

            current_force = force_function(previous_x)

            previous_x = update_positions(previous_x, current_force, beta, timestep, diffusion_coefficient)

            if configuration_output_frequency > 0:
                if step % configuration_output_frequency == 0:
                    trajectory.append(previous_x)

                    for state in states:
                        if state.is_in_state(previous_x): 
                            state_reached = True

            if state_reached: break

    return np.array(trajectory)

## 3. Generate trajectories

In the AIMMD framework, the training data are generated from a Transition Path Sampling (TPS) simulation. In a standard TPS calculation, one starts from an existing trajectory and selects a *shooting point*, that is, a configuration chosen from the trajectory according to a selection probability

$$
p_{\mathrm{sel}}(x \mid X),
$$

where $X$ denotes the current path and $x$ is one of the configurations belonging to it. New trajectories are then generated from the selected configuration and accepted or rejected according to a Monte Carlo criterion designed to sample the correct path ensemble. As a consequence, the sequence of trajectories produced by TPS forms a Markov chain in path space: each new trajectory depends on the previous one through the shooting move.

Implementing a complete TPS algorithm would considerably increase the complexity of the workshop and would shift the focus away from the machine-learning aspects that we wish to explore. Instead, we adopt a simplified strategy that reproduces the type of trajectory outcomes used by AIMMD while remaining easy to understand and implement.

Rather than selecting shooting points from previously generated trajectories, we sample configurations uniformly from a predefined region of configuration space. For the Wolfe–Quapp potential, this region corresponds to the square

$$
-2 \leq x \leq 2,
\qquad
-2 \leq y \leq 2,
$$

excluding the stable states. These randomly generated configurations play the role of shooting points in our simplified procedure.

For each shooting point, we then generate two independent trajectory segments. In standard TPS, one would typically integrate the dynamics both forward and backward in time from the selected configuration. Because we are using overdamped Langevin dynamics, however, there is no notion of velocity and therefore no natural distinction between forward and backward propagation. Instead, we simply generate two independent trajectories starting from the same shooting point and terminating when they reach either state A or state B.

Each shooting point therefore produces one of three possible outcomes:

$$
(n_A,n_B) \in {(2,0),(1,1),(0,2)},
$$

where $n_A$ and $n_B$ denote the number of trajectory segments that reach states A and B, respectively. Reactive shootings correspond to the outcome $(1,1)$, while the outcomes $(2,0)$ and $(0,2)$ are non-reactive.

A key feature of our approach is that we retain *all* generated trajectories. This mirrors the information available in a TPS simulation. In TPS, reactive trajectories are typically (there is still an acceptance criterion derived from detailed balance that reactive trajectories have to satisfy) accepted into the transition-path ensemble, whereas non-reactive trajectories correspond to rejected shooting moves. Although our procedure does not perform any acceptance or rejection step, keeping all outcomes allows us to construct the same type of labels used by AIMMD for training the committor model.

<div class="alert alert-block alert-danger">

It is important to emphasize that this simplified procedure does **not** generate the same statistical ensemble as TPS. In particular, two important differences should be kept in mind.

1. We do not know the distribution according to which the generated trajectories are sampled. In TPS, detailed balance together with the acceptance criterion guarantees that trajectories are distributed according to a well-defined path ensemble. Here, no acceptance rule or reweighting procedure is employed, and therefore there is no analogous guarantee. The generated trajectories should be viewed as a convenient source of labeled training data rather than as samples from a rigorously defined transition-path ensemble.

2. Trajectories generated from different shooting points are completely independent. In a TPS simulation, consecutive trajectories are correlated because each new trajectory is generated from the previous one through a shooting move. This correlation structure plays an important role in the statistical properties of TPS. In our simplified procedure, each shooting point is drawn independently and each pair of trajectory segments is generated independently of all others.
</div>

These approximations considerably simplify the implementation while preserving the essential ingredients needed for the machine-learning part of AIMMD. Most importantly, each shooting point is associated with the outcome of multiple stochastic trajectories, providing exactly the type of information required to train a model of the committor function.


### 3A. Generating Shooting Points

The following cell generates the initial set of shooting points that will be used to create trajectory data. Rather than selecting configurations from previously generated trajectories, as would be done in a standard TPS simulation, we sample configurations directly from configuration space.

The function `sample_shooting_points` implements a simple acceptance–rejection algorithm. Candidate configurations are drawn uniformly from the square

$$
-2 \leq x \leq 2,
\qquad
-2 \leq y \leq 2,
$$

and are accepted only if they do not belong to either of the stable states. If a sampled configuration falls inside state A or state B, it is discarded and a new configuration is generated. This procedure is repeated until the desired number of shooting points has been collected.

The resulting set of configurations provides a broad coverage of the transition region and the surrounding basins while avoiding points that are already committed to one of the stable states. In the present example, we generate `n_paths = 10` shooting points, which will subsequently be used as the starting configurations for the trajectory generation procedure.


### Implement Generation of Initial Shooting Points

<div class="alert alert-block alert-info">

**TASK**

1. Complete the function `sample_shooting_points` to generate `n_points` configurations uniformly distributed in the rectangular domain defined by `xmin`, `xmax`, `ymin`, and `ymax`.

2. Implement a simple acceptance-rejection procedure that discards any sampled configuration belonging to either stable state (`state_A` or `state_B`) and only stores accepted shooting points.

3. Return the accepted shooting points as a NumPy array of shape `(n_points, 2)`.

</div>


In [ ]:
n_paths = 1000

def sample_shooting_points(
    n_points: int,
    states: list,
    xmin: float = -2.0,
    xmax: float = 2.0,
    ymin: float = -2.0,
    ymax: float = 2.0,
):
    """
    Generate shooting points uniformly in a rectangular domain while
    excluding configurations belonging to the stable states.

    The function uses an acceptance-rejection algorithm. Candidate
    configurations are sampled uniformly from the rectangle

        [xmin, xmax] x [ymin, ymax]

    and are accepted only if they do not belong to any of the states
    provided in the list ``states``.

    Parameters
    ----------
    n_points : int
        Number of shooting points to generate.

    states : list
        List of state objects implementing the method
        ``is_in_state(point)``. A sampled configuration is rejected
        if it belongs to any of these states.

    xmin : float, optional
        Lower bound of the sampling domain along the x coordinate,
        by default -2.0.

    xmax : float, optional
        Upper bound of the sampling domain along the x coordinate,
        by default 2.0.

    ymin : float, optional
        Lower bound of the sampling domain along the y coordinate,
        by default -2.0.

    ymax : float, optional
        Upper bound of the sampling domain along the y coordinate,
        by default 2.0.

    Returns
    -------
    np.ndarray
        Array of accepted shooting points with shape
        ``(n_points, 2)`` and dtype ``np.float32``.
    """

    shooting_points = []

    ################################
    # Your code goes here
    ################################

    return np.asarray(shooting_points, dtype=np.float32)

shooting_points = sample_shooting_points(
    n_points=n_paths,
    states=[state_A, state_B]
)

### 3B. Generating Trajectory Data

The following cell generates the trajectory dataset that will be used throughout the remainder of the workshop. For each shooting point, two independent trajectory segments are propagated using the overdamped Langevin dynamics defined previously. Both trajectories start from the same configuration and are evolved until they reach either state A or state B.

Once the two trajectory segments have been generated, they are combined into a single continuous path by reversing the frames in the first branch and concatenating it with the second one. The resulting trajectory has the shooting point at its center and resembles the type of path that would be obtained from a TPS shooting move.

For each shooting point, we then record the outcome of the two trajectory segments. Specifically, we count how many trajectories terminate in state A and how many terminate in state B. This information is stored in the variables `nA` and `nB`, respectively. Because two trajectories are generated from every shooting point, the possible outcomes are

$$
(n_A,n_B)\in{(2,0),(1,1),(0,2)}.
$$

These counts play a central role in the AIMMD methodology. They provide a noisy estimate of the commitment behavior associated with the shooting point and will later be used to train the neural-network committor model.

The generated data are stored in four arrays:

* `paths`: the complete trajectories obtained by concatenating the two branches,
* `SP`: the shooting points from which the trajectories were launched,
* `nA`: the number of branches reaching state A,
* `nB`: the number of branches reaching state B.

Together, these quantities constitute the initial dataset from which we will learn an approximation of the committor function.


In [ ]:
max_path_length = 20000
configuration_output_frequency = 1
beta = 1.0
timestep = 0.001
diffusion_coefficient = 1.0

### Implement Construction of Transition Paths

<div class="alert alert-block alert-info">

**TASK**

1. Complete the definition of `full_path` by combining the two trajectory branches generated from the same shooting point into a single continuous trajectory.

2. Reverse one of the branches so that the shooting point becomes the connection point between the two segments.

3. Verify that the resulting trajectory starts and ends in stable states and passes through the shooting point only once.

**NOTE:** Both trajectory branches contain the shooting-point configuration. When concatenating the two segments, make sure that the shooting point is not included twice in the final trajectory.

</div>



In [ ]:
# Storage for the generated trajectories and their outcomes.
paths = []
SP = []
nA = []
nB = []

# Loop over all shooting points.
for shooting_point in tqdm(
    shooting_points,
    desc="Generating TPS shootings",
    unit="sp"
):

    # Generate the first trajectory branch starting from
    # the shooting point and ending in one of the stable states.
    branch_A = generate_path(
        shooting_point=shooting_point,
        force_function=wq.force,
        max_path_length=max_path_length,
        states=states,
        configuration_output_frequency=configuration_output_frequency,
        beta=beta,
        timestep=timestep,
        diffusion_coefficient=diffusion_coefficient
    )

    # Generate an independent trajectory branch from the
    # same shooting point.
    branch_B = generate_path(
        shooting_point=shooting_point,
        force_function=wq.force,
        max_path_length=max_path_length,
        states=states,
        configuration_output_frequency=configuration_output_frequency,
        beta=beta,
        timestep=timestep,
        diffusion_coefficient=diffusion_coefficient
    )

    # Construct a continuous trajectory by reversing one branch
    # and concatenating it with the other. 
    full_path = ... # Your code goes here

    # Store the complete trajectory.
    paths.append(full_path)

    # Count how many branches terminate in each stable state.
    count_A = 0
    count_B = 0

    for traj in (branch_A, branch_B):

        final_x = traj[-1]

        if state_A.is_in_state(final_x):
            count_A += 1

        elif state_B.is_in_state(final_x):
            count_B += 1

        else:
            raise RuntimeError(
                "Trajectory terminated outside stable states."
            )

    # Store the shooting point and the corresponding outcomes.
    #
    # Possible outcomes are:
    # (nA, nB) = (2, 0)  -> both branches reach A
    # (nA, nB) = (1, 1)  -> reactive shooting
    # (nA, nB) = (0, 2)  -> both branches reach B
    SP.append(shooting_point)
    nA.append(count_A)
    nB.append(count_B)

# Convert lists into NumPy arrays for later analysis
# and machine-learning training.
SP = np.array(SP)
nA = np.array(nA)
nB = np.array(nB)

### 3C. Inspecting the Generated Dataset

Before proceeding to the machine-learning stage, it is useful to examine the outcome of the shooting procedure. The table printed above summarizes the distribution of the trajectory outcomes in terms of the quantities $n_A$ and $n_B$, which count how many of the two trajectory branches terminate in states A and B, respectively. Because two trajectories are generated from each shooting point, only three outcomes are possible:

$$
(n_A,n_B)\in\{(2,0),(1,1),(0,2)\}.
$$

The outcomes $(2,0)$ and $(0,2)$ correspond to non-reactive shootings, where both trajectory segments commit to the same stable state. In contrast, the outcome $(1,1)$ corresponds to a reactive shooting, indicating that the chosen configuration lies close to the transition region separating the two basins. The relative frequency of these outcomes provides a first indication of how much of the sampled configuration space is relevant for the transition process.

To better understand the generated data, the next cell visualizes the trajectories and shooting points on top of the Wolfe–Quapp potential-energy surface. The contour plot shows the energy landscape together with the definitions of states A and B. A subset of the generated trajectories is displayed to illustrate how the dynamics explores the two metastable basins and occasionally crosses the barrier separating them.

The shooting points are colored according to their empirical commitment probability,

$$
p_B=\frac{n_B}{2},
$$

which can only take the values

$$
p_B\in\{0,\tfrac{1}{2},1\}.
$$

Points with $p_B=0$ (blue) correspond to shootings for which both trajectory segments reached state A, while points with $p_B=1$ (red) correspond to shootings for which both segments reached state B. The most interesting configurations are those with $p_B=\tfrac{1}{2}$ (yellow), for which one trajectory reaches A and the other reaches B. These configurations are located near the transition-state region and contain the most information about the committor function.

The figure provides an intuitive picture of the data that will be used for training. Configurations deep inside the basins are strongly committed to one state and therefore have committor values close to 0 or 1, whereas configurations near the barrier exhibit an intermediate commitment probability and carry information about the location of the transition region.


In [ ]:
n_reactive = np.sum((nA == 1) & (nB == 1))
n_AA = np.sum((nA == 2) & (nB == 0))
n_BB = np.sum((nA == 0) & (nB == 2))

print("=" * 50)
print("Trajectory Outcome Statistics")
print("=" * 50)
print(f"Total shooting points : {len(SP):5d}")
print()
print(f"(nA,nB) = (2,0) : {n_AA:5d}  (both branches reach A)")
print(f"(nA,nB) = (1,1) : {n_reactive:5d}  (reactive shootings)")
print(f"(nA,nB) = (0,2) : {n_BB:5d}  (both branches reach B)")
print()
print(f"Reactive fraction : {n_reactive/len(SP):.3f}")
print("=" * 50)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

max_plot_trajs = 50

levels = np.linspace(0, 20, 21)

contours = ax.contour(
    X,
    Y,
    V,
    levels=levels,
    colors="0.5",
    linewidths=1
)

filled = ax.contourf(
    X,
    Y,
    V,
    levels=levels,
    cmap="viridis",
    extend="max"
)

plt.colorbar(
    filled,
    ax=ax,
    label=r"$V(x,y)$ [$k_B T$]"
)


# ----- Plot states -----

for label, state in zip(["A", "B"], [state_A, state_B]):

    circle = Circle(
        state.center,
        radius=state.radius,
        facecolor=(1, 1, 1, 0.5),  # white, alpha=0.5
        edgecolor=(0, 0, 0, 1.0),  # black, alpha=1.0
        linewidth=2.5,
        zorder=20,
    )

    ax.add_patch(circle)

    ax.text(
        state.center[0],
        state.center[1],
        label,
        ha="center",
        va="center",
        fontsize=14,
        fontweight="bold",
        color="black",
        zorder=21,
    )

# ----- Plot trajectories -----

for traj in paths[:max_plot_trajs]:
    ax.plot(
        traj[:, 0],
        traj[:, 1],
        alpha=0.4,
        lw=1
    )

pB = nB / 2.0

mask_A = pB == 0.0
mask_TS = pB == 0.5
mask_B = pB == 1.0

ax.scatter(
    SP[mask_A, 0],
    SP[mask_A, 1],
    s=20,
    c="tab:blue",
    alpha=0.8,
    label=r"$p_B=0$",
    zorder=10,
)

ax.scatter(
    SP[mask_TS, 0],
    SP[mask_TS, 1],
    s=35,
    c="gold",
    edgecolor="black",
    linewidth=0.5,
    alpha=0.9,
    label=r"$p_B=0.5$",
    zorder=11,
)

ax.scatter(
    SP[mask_B, 0],
    SP[mask_B, 1],
    s=20,
    c="tab:red",
    alpha=0.8,
    label=r"$p_B=1$",
    zorder=10,
)
    
# -----------------------

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Wolfe–Quapp Potential")
ax.set_xlim(-2.2,2.2)
ax.set_ylim(-2.2,2.2)
ax.set_aspect("equal")
legend = ax.legend(loc="upper left")
legend.set_zorder(20)

plt.tight_layout()
plt.show()

## 4. Learning the Committor Function

Having generated a dataset of shooting points and trajectory outcomes, we now turn to the machine-learning component of the workshop. Our goal is to construct a model that approximates the committor function, that is, the probability that a configuration reaches state B before state A.

In principle, the committor is defined through an ensemble of stochastic trajectories initiated from a given configuration. Directly estimating this quantity for every point in configuration space would require a prohibitively large number of simulations. Instead, we seek to learn a continuous approximation of the committor from a finite set of trajectory data.

To achieve this, we will use a neural network implemented in PyTorch. The network takes a configuration $(x,y)$ as input and produces an estimate of the corresponding committor. The model parameters are determined by minimizing a loss function derived from the trajectory outcomes generated in the previous section. Intuitively, the loss encourages the network to assign low committor values to configurations whose trajectories predominantly reach state A, high committor values to configurations that predominantly reach state B, and intermediate values to configurations located in the transition region.

The machine-learning workflow consists of four main components:

1. **The neural-network model**, which provides a flexible parametric representation of the committor function.
2. **The loss function**, which quantifies the agreement between the network predictions and the observed trajectory outcomes.
3. **The dataset**, which organizes the shooting points and trajectory statistics into a format suitable for training with PyTorch.
4. **The training routine**, which iteratively updates the network parameters to minimize the loss.

Together, these components implement a supervised-learning problem in which the labels are not exact committor values but stochastic observations obtained from the trajectory ensemble. This distinction is important: the network is not trained to reproduce a known analytical function, but rather to infer the underlying committor from noisy dynamical data. In the following sections, we introduce each component in detail and explain how they combine to produce an accurate approximation of the committor function on the Wolfe–Quapp potential.


### 4A. Neural-Network Representation of the Committor

The central quantity that we wish to learn is the committor function,

$$
p_B(\mathbf{x}),
$$

which gives the probability that a trajectory initiated from configuration $\mathbf{x}$ reaches state B before state A. Since the exact committor is generally unknown, we represent it using a neural network whose parameters will be learned from the trajectory data generated in the previous section.

The class `CommittorNet` implements a simple feed-forward neural network. The input layer consists of two neurons corresponding to the Cartesian coordinates $(x,y)$ of the configuration. Because the Wolfe–Quapp system is two-dimensional, the Cartesian coordinates themselves provide a complete and convenient representation of the configuration. In realistic molecular systems, however, the dimensionality is much higher, and one typically employs atomic descriptors, collective variables, or other featurization schemes as inputs to the neural network. These inputs are passed through two hidden layers containing 16 neurons each, with hyperbolic tangent (`Tanh`) activation functions. Finally, a single output neuron produces a scalar value $q$:

$$
q = f_\theta(x,y),
$$

where $f_\theta$ denotes the neural network with parameters $\theta$.

An important aspect of the implementation is that the network does not directly output a probability. Instead, the `forward` method returns the unconstrained scalar quantity $q$, often referred to as the *logit*. Because $q$ can take any real value, it is easier to optimize during training than a quantity restricted to the interval $[0,1]$.

To obtain a valid committor probability, the class provides the method

```python
model.committor(x)
```

which applies the sigmoid transformation

$$
p_B(x)
=
\sigma(q)
=
\frac{1}{1+e^{-q}}.
$$

This mapping guarantees that the predicted committor satisfies

$$
0 \leq p_B(x) \leq 1.
$$

Configurations deep inside state A are therefore expected to have large negative values of $q$, corresponding to committor values close to zero, while configurations deep inside state B should have large positive values of $q$, corresponding to committor values close to one. Configurations located near the transition-state region are expected to have

$$
q \approx 0,
$$

which implies

$$
p_B \approx \frac{1}{2}.
$$

This interpretation of the network output is particularly important for the AIMMD methodology. The quantity $q$ acts as a reaction coordinate whose zero level set approximates the transition-state surface, while the sigmoid-transformed output provides a probabilistic estimate of the committor itself.

Although the architecture used here is intentionally small, it is more than sufficient for the two-dimensional Wolfe–Quapp system. In realistic molecular applications, the same ideas are employed with larger neural networks and high-dimensional molecular descriptors, but the underlying principles remain unchanged.


### Implementing the Committor Neural Network

<div class="alert alert-block alert-info">

**TASK**

1. Complete the constructor of the `CommittorNet` class by defining a fully connected neural network that maps a two-dimensional input configuration to a single scalar output.

2. Use two hidden layers containing 16 neurons each and the hyperbolic tangent (`Tanh`) activation function between successive layers.

3. Complete the `forward` method so that it evaluates the neural network and returns the reaction coordinate $q(\mathbf{x})$.

4. Complete the `committor` method so that it returns the committor probability

   $$
   p_B(\mathbf{x})
   =
   \sigma\!\left(q(\mathbf{x})\right),
   $$

   where $\sigma$ denotes the sigmoid function.

**NOTE:** The output of the neural network should be a single unconstrained scalar (q). The sigmoid transformation should only be applied in the `committor` method, not in the `forward` method.

</div>


In [ ]:
class CommittorNet(nn.Module):
    """
    Neural-network model for approximating the committor function.

    The network takes a two-dimensional configuration
    x = (x, y) as input and returns a scalar reaction coordinate
    q(x), often referred to as the committor logit. The committor
    probability is obtained by applying a sigmoid transformation,

        p_B(x) = sigmoid(q(x)),

    which maps the output to the interval [0, 1].

    The architecture consists of two hidden layers with 16 neurons
    each and hyperbolic tangent activation functions. Although
    relatively small, this network is sufficiently expressive to
    learn the committor function of the Wolfe--Quapp potential.

    Methods
    -------
    forward(x)
        Evaluate the network and return the reaction coordinate
        q(x).

    committor(x)
        Evaluate the corresponding committor probability
        p_B(x) = sigmoid(q(x)).

    Notes
    -----
    In the AIMMD framework, the raw network output q(x) is used as
    the learned reaction coordinate. Configurations satisfying
    q(x) = 0 correspond to p_B(x) = 0.5 and approximate the
    transition-state ensemble.
    """
    def __init__(self):

        super().__init__()

        ################################
        # Your code goes here
        ################################

    def forward(self, x):

        ################################
        # Your code goes here
        ################################

        return q

    def committor(self, x):

        ################################
        # Your code goes here
        ################################
 
        return pB

### 4B. Loss Function

Having defined a neural-network representation of the committor, we now need a way to determine the optimal network parameters from the trajectory data. This is achieved through a loss function, which quantifies the agreement between the predictions of the neural network and the observed outcomes of the shooting trajectories.

For a given shooting point, the neural network predicts a committor value

$$
p_B(\mathbf{x}),
$$

which can be interpreted as the probability that a trajectory initiated from configuration $\mathbf{x}$ reaches state B before state A. In our dataset, however, we do not know the exact committor. Instead, we observe the outcomes of two independent trajectory segments and record the numbers $n_A$ and $n_B$ of trajectories that terminate in states A and B.

Assuming that each trajectory represents an independent Bernoulli trial with success probability $p_B$, the probability of observing a particular outcome $(n_A,n_B)$ is proportional to

$$
p_B^{\,n_B}(1-p_B)^{\,n_A}.
$$

More generally, for $n_A+n_B$ independent trajectories, the full probability distribution is given by the binomial distribution

$$
P(n_A,n_B \mid p_B)
=
\binom{n_A+n_B}{n_B}
p_B^{\,n_B}
(1-p_B)^{\,n_A}
$$

The AIMMD loss is obtained by maximizing the likelihood of the observed trajectory outcomes, or equivalently by minimizing the negative log-likelihood. Taking the negative logarithm of the expression above yields

$$
-\log P(n_A,n_B \mid p_B)
=
-\log\binom{n_A+n_B}{n_B}
-
n_B\log p_B
-
n_A\log(1-p_B).
$$

The binomial coefficient does not depend on the neural-network parameters and therefore has no effect on the optimization. It can consequently be omitted from the loss function, leading to

$$
\mathcal{L}
=
-
\Big[
n_B\log p_B
+
n_A\log(1-p_B)
\Big].
$$

This is precisely the quantity implemented in the function `committor_loss`.

For numerical stability, a small constant `eps` is added inside the logarithms to avoid evaluating $\log(0)$ when the predicted committor is extremely close to zero or one. The loss is evaluated for every shooting point in a minibatch and the average value is returned.

An intuitive interpretation of this expression is straightforward. If a shooting point predominantly generates trajectories ending in state B, the loss is minimized when the network predicts a large committor value. Conversely, if most trajectories terminate in state A, the loss favors small committor values. Configurations producing mixed outcomes, such as $(n_A,n_B)=(1,1)$, naturally encourage predictions close to $p_B=\tfrac{1}{2}$ and therefore help identify the transition-state region.

The loss function is therefore directly connected to the stochastic dynamics of the system. Rather than learning from externally provided labels, the neural network learns from the observed outcomes of trajectory shootings and gradually constructs a probabilistic model of the committor function.


### Implementing the Committor Loss Function

<div class="alert alert-block alert-info">

**TASK**

1. Evaluate the committor probability predicted by the neural network for the input configurations `x`.

2. Implement the negative log-likelihood associated with the observed trajectory outcomes `(nA, nB)`:

   $$
   \mathcal{L}
   =
   -n_B \log(p_B)
   -
   n_A \log(1-p_B).
   $$

3. Include the small constant `eps` inside the logarithms to avoid numerical instabilities when the predicted committor is very close to 0 or 1.

4. Return the mean loss over all shooting points in the batch.

**NOTE:** The loss should be computed using the committor probability $p_B$ returned by the network, not the raw reaction coordinate $q$.

</div>


In [ ]:
def committor_loss(model, x, nA, nB, eps=1e-8):
    """
    Negative log-likelihood loss for committor learning.

    The loss is derived from the likelihood of observing the
    trajectory outcomes associated with each shooting point.
    Given a predicted committor probability

        p_B(x) = model.committor(x),

    and counts nA and nB corresponding to the numbers of
    trajectories reaching states A and B, respectively, the
    likelihood of the observations is modeled using a binomial
    distribution. Neglecting the combinatorial prefactor, the
    resulting loss for a single shooting point is

        L = -n_B log(p_B) - n_A log(1 - p_B).

    The function returns the mean loss over all shooting points
    in the batch.

    Parameters
    ----------
    model : nn.Module
        Neural-network model providing the method
        ``committor(x)``.

    x : torch.Tensor
        Batch of shooting-point configurations with shape
        ``(batch_size, n_features)``.

    nA : torch.Tensor
        Number of trajectories associated with each shooting
        point that terminate in state A.

    nB : torch.Tensor
        Number of trajectories associated with each shooting
        point that terminate in state B.

    eps : float, optional
        Small numerical constant added inside the logarithms to
        avoid evaluating ``log(0)``, by default ``1e-8``.

    Returns
    -------
    torch.Tensor
        Mean negative log-likelihood over the batch.

    Notes
    -----
    For the two-way shooting procedure used in this workshop,
    the possible outcomes are

        (n_A, n_B) ∈ {(2,0), (1,1), (0,2)}.

    Consequently, the loss encourages the network to predict
    committor values close to 0 for configurations committed to
    state A, close to 1 for configurations committed to state B,
    and close to 0.5 for configurations in the transition-state
    region.
    """

    ################################
    # Your code goes here
    ################################

    return loss

### 4C. Preparing the Training Dataset

Before training the neural network, the trajectory data must be organized into a format that can be efficiently processed by PyTorch. This is the role of the `ShootingPointsDataset` class, which serves as a bridge between the data generated by the simulations and the training routines used by the machine-learning framework. If needed check the PyTorch documentation [here](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html).

The dataset is constructed from three arrays:

* `SP`, containing the shooting-point coordinates,
* `nA`, containing the number of trajectories that reached state A,
* `nB`, containing the number of trajectories that reached state B.

For each shooting point $\mathbf{x}^{\mathrm{sp}}_i$, the dataset therefore stores the triplet

$$
\left(
\mathbf{x}^{\mathrm{sp}}_i,
n_{A,i},
n_{B,i}
\right),
$$

which contains both the input configuration and the corresponding trajectory outcome statistics.

During initialization, the NumPy arrays are converted into PyTorch tensors with floating-point precision. This allows the data to be used directly in the neural-network computations and enables efficient execution on either CPUs or GPUs.

The class implements the two methods required by PyTorch datasets:

* `__len__`, which returns the total number of shooting points in the dataset,
* `__getitem__`, which returns the data associated with a particular index.

When an element of the dataset is requested, the method returns

$$
(\mathbf{x}^{\mathrm{sp}}_i,n_{A,i},n_{B,i}),
$$

corresponding to a single training example.

Although this dataset class is very simple, it plays an important role in separating data management from model training. The neural network and loss function only need to know how to process minibatches of shooting points and trajectory counts, while the dataset provides a clean and reusable interface to the underlying simulation data.


In [ ]:
class ShootingPointsDataset(Dataset):
    """
    PyTorch dataset containing shooting points and their
    corresponding trajectory outcomes.

    Each sample in the dataset consists of a shooting point
    together with the numbers of trajectory branches reaching
    states A and B,

    $$
    (\mathbf{x}, n_A, n_B).
    $$

    These quantities provide the training data for committor
    learning. Rather than supplying target committor values,
    the dataset stores the observed outcomes of the shooting
    procedure. The committor network is then trained by
    maximizing the likelihood of these observations through
    the committor loss function.

    Parameters
    ----------
    SP : np.ndarray
        Array of shooting-point coordinates with shape
        ``(n_samples, 2)``.

    nA : np.ndarray
        Number of trajectory branches terminating in state A
        for each shooting point.

    nB : np.ndarray
        Number of trajectory branches terminating in state B
        for each shooting point.

    Notes
    -----
    In the present workshop, two trajectory branches are
    generated from each shooting point. Consequently, the
    possible outcomes are

    $$
    (n_A, n_B)
    \in
    \{(2,0), (1,1), (0,2)\}.
    $$

    The reactive case

    $$
    (n_A, n_B) = (1,1)
    $$

    corresponds to one branch reaching state A and the other
    reaching state B.

    Methods
    -------
    __len__()
        Return the number of shooting points stored in the
        dataset.

    __getitem__(idx)
        Return the tuple

        $$
        (\mathbf{x}_i, n_{A,i}, n_{B,i})
        $$

        corresponding to the sample with index ``idx``.
    """
    def __init__(self, SP, nA, nB):

        # Store the shooting-point coordinates as a tensor.
        # Each row corresponds to a configuration from which
        # two trajectory branches have been generated.
        self.SP = torch.tensor(
            SP,
            dtype=torch.float32
        )

        # Store the number of branches terminating in state A
        # for each shooting point.
        self.nA = torch.tensor(
            nA,
            dtype=torch.float32
        )

        # Store the number of branches terminating in state B
        # for each shooting point.
        self.nB = torch.tensor(
            nB,
            dtype=torch.float32
        )

    def __len__(self):

        # Return the total number of shooting points contained
        # in the dataset.
        return len(self.SP)

    def __getitem__(self, idx):

        # Return a single training example consisting of:
        #
        # - the shooting-point coordinates
        # - the number of trajectories reaching state A
        # - the number of trajectories reaching state B
        #
        # These quantities are used by the committor loss
        # to construct the likelihood of the observed outcomes.
        return (
            self.SP[idx],
            self.nA[idx],
            self.nB[idx]
        )

# Create the dataset used for training the committor network.
dataset = ShootingPointsDataset(SP, nA, nB)

### 4D. Training the Committor Network

With the neural-network model, loss function, and dataset in place, the final component of the machine-learning pipeline is the training procedure. The goal of training is to determine the network parameters that minimize the negative log-likelihood loss introduced in the previous section and thereby produce the best possible approximation of the committor function.

The function `train_committor` implements a standard supervised-learning workflow using PyTorch. The dataset is first wrapped in a `DataLoader`, which automatically partitions the data into minibatches and shuffles the samples at the beginning of each epoch. Shuffling helps prevent correlations in the ordering of the training data from influencing the optimization process.

The network parameters are optimized using the Adam algorithm. Starting from a random initialization of the network weights, the optimizer iteratively updates the parameters to reduce the loss.

During each training epoch, the following steps are performed:

1. A minibatch of shooting points and trajectory counts is retrieved from the `DataLoader`.
2. The data are transferred to the selected computational device (CPU or GPU).
3. The gradients from the previous optimization step are reset.
4. The committor loss is evaluated for the current minibatch.
5. Backpropagation is used to compute the gradient of the loss with respect to all network parameters.
6. The optimizer updates the parameters using these gradients.

Mathematically, the parameter update can be viewed as an iterative optimization of

$$
\theta^{(k+1)}
=
\theta^{(k)}
-
\eta
\nabla_\theta \mathcal{L},
$$

where $\theta$ denotes the collection of network parameters, $\mathcal{L}$ is the loss function, and $\eta$ is an effective learning rate determined by the Adam optimizer.

At the end of each epoch, the average loss over all minibatches is computed and stored in the list `history`. Monitoring this quantity provides a simple way to assess the convergence of the training procedure. A steadily decreasing loss indicates that the neural network is becoming increasingly consistent with the trajectory outcomes observed in the dataset.

The function ultimately returns the complete training history, which can be visualized to verify that the optimization has converged successfully. Once training is complete, the network can be evaluated on arbitrary configurations and used as a continuous approximation of the committor function across the entire Wolfe–Quapp potential-energy surface.

In [ ]:
def train_committor(
    model,
    dataset,
    epochs=200,
    batch_size=128,
    lr=1e-3,
    device="cpu"
):
    """
    Train a neural-network committor model.

    The training procedure minimizes the committor loss over the
    provided dataset using mini-batch stochastic optimization.
    The model parameters are updated using the Adam optimizer.

    For each mini-batch, the network predicts committor values
    for the shooting points and the corresponding negative
    log-likelihood loss is computed from the observed trajectory
    outcomes $(n_A, n_B)$. The gradients of this loss are then
    used to update the network parameters.

    Parameters
    ----------
    model : nn.Module
        Neural-network committor model.

    dataset : Dataset
        Dataset containing tuples

        $$
        (\mathbf{x}, n_A, n_B)
        $$

        where $\mathbf{x}$ is a shooting point and
        $(n_A, n_B)$ are the corresponding trajectory outcomes.

    epochs : int, optional
        Number of training epochs, by default 200.

    batch_size : int, optional
        Number of samples per mini-batch, by default 128.

    lr : float, optional
        Learning rate used by the Adam optimizer,
        by default $10^{-3}$.

    device : str, optional
        Device on which the model is trained
        (e.g. ``"cpu"`` or ``"cuda"``),
        by default ``"cpu"``.

    Returns
    -------
    list
        List containing the average training loss at the end
        of each epoch.

    Notes
    -----
    The returned loss history can be used to monitor
    convergence of the optimization and to detect potential
    underfitting or overfitting of the model.
    """
    # Create a data loader that will iterate over the
    # dataset in random mini-batches.
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True
    )

    # Initialize the Adam optimizer.
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr
    )

    # Move the model to the selected device.
    model.to(device)

    # Store the average loss at each epoch.
    history = []

    # Main training loop.
    for epoch in tqdm(
        range(epochs),
        desc="Training committor"
    ):

        epoch_loss = 0.0

        # Iterate over all mini-batches.
        for sp, nA, nB in loader:

            # Move the current batch to the selected device.
            sp = sp.to(device)
            nA = nA.to(device)
            nB = nB.to(device)

            # Reset gradients from the previous optimization step.
            optimizer.zero_grad()

            # Compute the committor loss for the current batch.
            loss = committor_loss(
                model,
                sp,
                nA,
                nB
            )

            # Compute gradients of the loss with respect to
            # all trainable model parameters.
            loss.backward()

            # Update the model parameters.
            optimizer.step()

            # Accumulate the batch loss.
            epoch_loss += loss.item()

        # Compute the average loss over all mini-batches.
        epoch_loss /= len(loader)

        # Store the loss for later analysis.
        history.append(epoch_loss)

    return history

### 4E. Training the Committor Model

We are now ready to train the neural network using the trajectory data generated in the previous sections. First, an instance of the `CommittorNet` model is created. At this stage, the network parameters are initialized randomly and therefore the predicted committor values have no physical meaning.

The training process is initiated by calling the `train_committor` function. The previously constructed dataset is provided together with the main hyperparameters controlling the optimization procedure:

* `epochs=200`: the number of complete passes through the training dataset,
* `batch_size=128`: the number of samples processed simultaneously during each optimization step,
* `lr=1e-3`: the learning rate used by the Adam optimizer.

During training, the network repeatedly compares its predictions with the trajectory outcomes stored in the dataset and adjusts its parameters to minimize the committor loss. The optimization progressively transforms the initially random mapping into a function that approximates the probability of reaching state B before state A.

The function returns the variable `history`, which contains the average loss value recorded after each training epoch. This information can be used to monitor the convergence of the optimization process and verify that the network is successfully learning from the trajectory data.


In [ ]:
model = CommittorNet()

history = train_committor(
    model,
    dataset,
    epochs=200,
    batch_size=128,
    lr=1e-3
)

#### Monitoring the Training Process

A useful diagnostic of the optimization procedure is the evolution of the loss function during training. The following cell plots the average loss recorded at the end of each epoch.

Because the loss corresponds to the negative log-likelihood of the observed trajectory outcomes, decreasing values indicate that the neural network is becoming increasingly consistent with the data. In the early stages of training, the loss typically decreases rapidly as the network learns the broad distinction between configurations committed to states A and B. As training progresses, the decrease becomes slower and eventually reaches a plateau, signaling that the optimization has converged.

Although the loss itself does not have a direct physical interpretation, its behavior provides a valuable indication of whether the training procedure is functioning correctly. A smooth and steadily decreasing curve is generally a sign of successful optimization, whereas strong oscillations or increasing values may indicate inappropriate hyperparameter choices or insufficient training data.


In [ ]:
plt.figure(figsize=(6,4))

plt.plot(history)

plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.tight_layout()
plt.show()

### 4F. Visualizing the Learned Committor Function

After training, the neural network provides a continuous approximation of the committor function throughout the entire configuration space. To assess the quality of the learned model and understand how it relates to the underlying dynamics, we evaluate the network on a dense two-dimensional grid covering the Wolfe–Quapp potential.

The resulting figure combines information about both the energy landscape and the learned committor. The background color map and gray contour lines represent the potential-energy surface, while the white contour lines correspond to iso-committor surfaces predicted by the neural network. Each contour is labeled with its associated committor value,

$$
p_B(\mathbf{x}),
$$

which can be interpreted as the probability that a trajectory initiated from configuration $\mathbf{x}$ reaches state B before state A.

Several contour levels are displayed between $p_B=0$ and $p_B=1$, together with additional contours at $p_B=0.02$, $0.05$, $0.95$, and $0.98$. These additional levels help resolve the regions close to the stable states, where the committor changes rapidly from intermediate values to nearly complete commitment to one basin.

Particular attention should be paid to the contour corresponding to

$$
p_B=\frac{1}{2},
$$

which is drawn with a thicker line than the other iso-committors. This contour is of special importance because it approximates the transition-state surface. Configurations located on this surface have equal probability of reaching either state and therefore represent the dynamical bottleneck of the transition process. In the language introduced earlier, these are precisely the configurations for which the neural-network logit satisfies

$$
q \approx 0.
$$

For the Wolfe–Quapp system, the learned committor can be directly compared with the shape of the energy landscape. The figure therefore provides an intuitive validation of the machine-learning model: committor values close to zero are found near state A, committor values close to one near state B, and the $p_B=0.5$ contour naturally follows the saddle region through which reactive trajectories pass. This demonstrates that the neural network has successfully extracted the essential dynamical information from the trajectory data and reconstructed a physically meaningful approximation of the committor function.


In [ ]:
# Grid
x = np.linspace(-2.25, 2.25, 400)
y = np.linspace(-2.25, 2.25, 400)

X, Y = np.meshgrid(x, y)

coords = np.stack(
    [X.ravel(), Y.ravel()],
    axis=1
)

# Evaluate energy
V = wq.energy(coords)

V = V.reshape(X.shape)

fig, ax = plt.subplots(figsize=(8, 6))

levels = np.linspace(0, 20, 21)

contours = ax.contour(
    X,
    Y,
    V,
    levels=levels,
    colors="0.5",
    linewidths=1
)

filled = ax.contourf(
    X,
    Y,
    V,
    levels=levels,
    cmap="viridis",
    extend="max"
)

plt.colorbar(
    filled,
    ax=ax,
    label=r"$V(x,y)$ [$k_B T$]"
)

with torch.no_grad():

    pB = model.committor(
        torch.tensor(
            coords,
            dtype=torch.float32
        )
    )

pB = pB.numpy().reshape(X.shape)

levels = np.linspace(0, 1, 11)
levels = levels[~np.isclose(levels, 0.5)]
levels = np.sort(
    np.concatenate([
        levels,
        [0.02, 0.05, 0.95, 0.98]
    ])
)

committor_contours = ax.contour(
    X,
    Y,
    pB,
    levels=levels,
    colors="white",
    linewidths=1,
    alpha=0.7,
    zorder=30
)

ax.clabel(
    committor_contours,
    inline=True,
    fontsize=8,
    fmt="%.2f"
)

ts_contours = ax.contour(
    X,
    Y,
    pB,
    levels=[0.5],
    colors="white",
    linewidths=4,
    zorder=31
)

ax.clabel(
    ts_contours,
    inline=True,
    fontsize=8,
    fmt="%.1f"
)

# ----- Plot states -----

for label, state in zip(["A", "B"], [state_A, state_B]):

    circle = Circle(
        state.center,
        radius=state.radius,
        facecolor=(1,1,1,.5),
        edgecolor=(0,0,0,1),
        linewidth=2.5,
        zorder=20,
    )

    ax.add_patch(circle)

    ax.text(
        state.center[0],
        state.center[1],
        label,
        ha="center",
        va="center",
        fontsize=14,
        fontweight="bold",
        color="black",
        zorder=21,
    )

    
# -----------------------

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_xlim(-2.2,2.2)
ax.set_ylim(-2.2,2.2)
ax.set_aspect("equal")
# ax.legend(loc="upper left")

plt.tight_layout()
plt.show()

### 4G. Confusion plot

While visual inspection of the committor contours provides a qualitative assessment of the model, a more rigorous validation can be obtained by comparing the predicted committor values against independent reference calculations obtained from a large number of independent fleeting trajectories. The reference dataset contains the shooting-point coordinates, the estimated committor value, and its statistical uncertainty. Since generating these reference values is computationally expensive, they are provided as a precomputed dataset.

In [ ]:
reference = np.load(DATA_DIR.joinpath("WQ_potential/reference_committor/reference_committor_beta_1d0_outfreq_1_dt_0d001_D_1d0.npz"))

shooting_points_ref = reference["shooting_points"]
pB_ref = reference["pB_ref"]
sigma_ref = reference["sigma_ref"]

print(f"Loaded {len(shooting_points_ref)} reference points")


The figure below shows a confusion plot between the neural-network predictions and the reference committor. Each point corresponds to a shooting point included in the benchmark dataset, while the horizontal error bars indicate the statistical uncertainty associated with the reference estimate. Perfect agreement would place all points on the diagonal line

$$
p_B^{\mathrm{NN}}
=
p_B^{\mathrm{ref}}.
$$

Deviations from this line reveal regions where the model is less accurate, whereas points lying within the uncertainty bounds indicate agreement within the statistical error of the reference calculation. For the Wolfe-Quapp system, the parity plot typically shows that even a relatively small neural network trained on a modest amount of trajectory data can recover the committor with remarkable accuracy across the entire transition region.

The uncertainty of the reference committor is estimated assuming a binomial distribution of trajectory outcomes,

$$
\sigma(p_B)
=
\sqrt{
\frac{p_B(1-p_B)}
     {n_A+n_B}
},
$$

where $n_A$ and $n_B$ denote the numbers of fleeting trajectories terminating in states A and B, respectively.

As a first thing we evaluate the model on the reference dataset:

### Evaluate the Learned Committor on the Reference Dataset

<div class="alert alert-block alert-info">

**TASK**

1. Use the trained neural network to evaluate the committor probability

   $$
   p_B(\mathbf{x})
   $$

   for all configurations in the reference dataset.

</div>


In [ ]:
# Evaluate the neural-network committor no the reference dataset

with torch.no_grad():

    ##############################
    # Your code goes here
    ##############################

pB_pred = (
    pB_pred
    .squeeze(-1)
    .cpu()
    .numpy()
)

Then we plot $p_B^{\mathrm{NN}}$ vs. $p_B^{\mathrm{ref}}$

In [ ]:
# --------------------------------------------------
# Confusion plot
# --------------------------------------------------

fig, ax = plt.subplots(
    figsize=(6, 6)
)

ax.errorbar(
    pB_ref,
    pB_pred,
    xerr=sigma_ref,
    fmt="o",
    ms=4,
    alpha=0.7,
    capsize=2,
)

ax.plot(
    [0, 1],
    [0, 1],
    "k--",
    lw=2,
    label="Perfect agreement"
)

ax.set_xlabel(
    r"Reference committor $p_B^{\mathrm{ref}}$"
)

ax.set_ylabel(
    r"Predicted committor $p_B^{\mathrm{NN}}$"
)

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

ax.set_aspect("equal")

ax.legend()

plt.tight_layout()
plt.show()

## 5. Using the Learned Committor to Guide Shooting-Point Selection

A central idea of the AIMMD methodology is that the learned committor can be used not only to characterize the transition mechanism, but also to improve the efficiency of the sampling procedure itself. Once a neural-network approximation of the committor has been obtained, the model can be used to identify configurations that are likely to lie close to the transition-state region. Future shooting points can then be selected preferentially from these configurations, focusing computational effort on the most informative parts of configuration space.

In a full AIMMD simulation, this procedure is embedded within an iterative self-consistent loop. Starting from an initial Transition Path Sampling simulation, a neural network is trained to approximate the committor. The resulting model is then used to define a shooting-point selection probability along the sampled trajectories. New TPS trajectories are generated from the selected shooting points, incorporated into the training set, and the committor model is retrained. Repeating this cycle progressively concentrates the sampling effort near the transition-state ensemble and improves the quality of the learned committor.

In this workshop, we adopt a simplified approach. Rather than performing additional rounds of trajectory generation and retraining, we focus on the role of the learned committor in defining the shooting-point selection probability. Specifically, we use the trajectories generated in our initial sampling procedure and restrict our attention to the reactive trajectories, that is, those for which one branch reaches state A and the other reaches state B. These trajectories contain the most relevant information about the transition process and play a role analogous to the transition paths sampled in TPS.

We then evaluate the learned reaction coordinate $q$ along the configurations belonging to these reactive trajectories and use it to construct a shooting-point selection probability. Following AIMMD, configurations with values of $q$ close to zero are assigned a higher probability of being selected. Since the condition

$$
q=0
$$

corresponds to a committor value

$$
p_B=\frac{1}{2},
$$

the resulting selection procedure naturally concentrates around the transition-state region.

It is important to emphasize that this differs from the full AIMMD workflow in two respects. First, we do not generate a new generation of trajectories from the selected configurations. Second, we do not retrain the neural network using additional data. Consequently, no self-consistent learning loop is performed. Instead, our goal is to visualize how the learned committor would bias the shooting-point distribution and to understand how the parameter controlling the selection probability affects the localization of the sampled configurations.

Although simplified, this analysis captures the key physical insight behind AIMMD: once a reasonable approximation of the committor is available, it can be used to automatically identify and target the transition-state region, thereby focusing computational effort on the configurations that are most informative for rare-event sampling.


### 5A. Defining the Shooting-Point Selection Probability

The first step is to use the learned reaction coordinate to construct a probability distribution for selecting new shooting points. In the AIMMD framework, this distribution should favor configurations located near the transition-state region, where the trajectories contain the most information about the committor function.

Recall that the neural network produces a reaction coordinate $q$, with the transition-state surface corresponding approximately to

$$
q = 0.
$$

Configurations with large positive or negative values of $q$ are already strongly committed to one of the two stable states and are therefore less informative for future shooting moves. Consequently, the selection probability should be maximal at $q=0$ and decrease as $|q|$ increases.

Following the AIMMD approach, we use a Lorentzian (or Cauchy) distribution,

$$
w[q(x)]
=
\frac{1}{1+\left(\dfrac{q(x)}{\gamma}\right)^2},
$$

where $\gamma$ is a width parameter controlling how strongly the selection is focused around the transition state. The normalized shooting-point selection probability is then obtained as

$$
p_{\mathrm{sel}}(x|X)
=
\frac{w[q(x)]}
{\sum_{x'\in X} w[q(x')]},
$$

where the sum runs over all configurations belonging to a given reactive trajectory.

The parameter $\gamma$ determines the breadth of the selection region:

* Large values of $\gamma$ produce a broad distribution and therefore a nearly uniform selection along the trajectory.
* Small values of $\gamma$ strongly concentrate the probability around configurations with $q \approx 0$.
* In the limit $\gamma \rightarrow 0$, only configurations very close to the transition-state surface are selected with significant probability.

The function `selection_probability` implements this procedure. Given a set of reaction-coordinate values **evaluated along a trajectory**, it computes the Lorentzian weights and normalizes them to obtain a valid probability distribution. These probabilities can then be used to randomly select new shooting points, mimicking the adaptive shooting strategy employed by AIMMD.

An attractive feature of the Lorentzian distribution is its relatively heavy tails. Compared to a Gaussian distribution centered at $q=0$, it still assigns a non-negligible probability to configurations somewhat removed from the transition state. This prevents the sampling from becoming overly localized and helps maintain diversity in the selected shooting points while still preferentially targeting the most informative regions of configuration space.


### Implement the Lorentzian Shooting-Point Selection Probability

<div class="alert alert-block alert-info">

**TASK**

1. Given a set of reaction-coordinate values `q`, compute the Lorentzian weights

   $$
   w_i
   =
   \frac{1}
   {1+\left(q_i/\gamma\right)^2}.
   $$

2. Normalize the weights to obtain a valid probability distribution

   $$
   p_i
   =
   \frac{w_i}
   {\sum_j w_j},
   $$

   such that all probabilities are non-negative and sum to one.

</div>


In [ ]:
def lorentzian_selection_probability(q, gamma=1.0):
    """
    Compute the Lorentzian shooting-point selection probability.

    This function implements the shooting-point selection strategy
    used in AIMMD. Given a set of reaction-coordinate values
    evaluated along a trajectory,

        q = (q_1, q_2, ..., q_N),

    each configuration is assigned a Lorentzian weight

        w_i = 1 / (1 + (q_i / gamma)^2),

    where ``gamma`` controls the width of the distribution.
    Configurations with reaction-coordinate values close to
    ``q = 0`` receive the largest weights and are therefore
    more likely to be selected as future shooting points.

    The weights are subsequently normalized to obtain a valid
    probability distribution,

        p_i = w_i / sum_j w_j,

    which can be used together with ``numpy.random.choice`` to
    sample configurations along a trajectory.

    Parameters
    ----------
    q : np.ndarray
        One-dimensional array containing the reaction-coordinate
        values evaluated along a trajectory.

    gamma : float, optional
        Width parameter of the Lorentzian distribution. Smaller
        values concentrate the selection probability more strongly
        around ``q = 0``, while larger values lead to a broader,
        nearly uniform selection. By default 1.0.

    Returns
    -------
    np.ndarray
        Normalized selection probabilities with the same shape as
        ``q``. The returned probabilities are non-negative and sum
        to one.

    """

    ################################
    # Your code goes here
    ################################

    return p_i

### 5B. Evaluating the Learned Reaction Coordinate Along a Trajectory

To construct the shooting-point selection probability, we must first evaluate the learned reaction coordinate along the configurations of a reactive trajectory. The function `evaluate_q` performs this task.

Given a trajectory represented as an array of configurations,

$$
X = {\mathbf{x}_1,\mathbf{x}_2,\ldots,\mathbf{x}_N},
$$

the function evaluates the neural network on every configuration and returns the corresponding values of the reaction coordinate,

$$
q_i = f_\theta(\mathbf{x}_i),
\qquad i=1,\ldots,N.
$$

These values measure the position of each configuration relative to the transition-state region. Configurations with large negative values of $q$ are associated with state A, configurations with large positive values of $q$ are associated with state B, and configurations with

$$
q \approx 0
$$

lie near the transition state.

The output of this function is a one-dimensional array containing the reaction-coordinate values associated with every configuration along the trajectory,

$$
(q_1,q_2,\ldots,q_N).
$$

These values will be used in the next step to compute the Lorentzian selection probability and identify the configurations that are most likely to be chosen as new shooting points.


### Evaluate the Learned Reaction Coordinate Along a Trajectory

<div class="alert alert-block alert-info">

**TASK**

1. Evaluate the neural-network model on all trajectory configurations to obtain the corresponding reaction-coordinate values

   $$
   q(\mathbf{x}).
   $$

**NOTE:** Remember to convert the result into a one-dimensional NumPy array and before returning it.

</div>


In [ ]:
def evaluate_q(model, trajectory):
    """
    Evaluate the learned reaction coordinate along a trajectory.

    The function applies the neural-network model to every
    configuration contained in a trajectory and returns the
    corresponding values of the reaction coordinate

        q(x) = model(x).

    The evaluation is performed without tracking gradients,
    since the network is used only for inference.

    Parameters
    ----------
    model : nn.Module
        Trained committor model. The forward pass of the model
        must return the reaction coordinate q.

    trajectory : np.ndarray
        Array containing the configurations of a trajectory.
        Expected shape is ``(n_frames, n_features)``, where
        ``n_features = 2`` for the Wolfe--Quapp system.

    Returns
    -------
    np.ndarray
        One-dimensional array containing the reaction-coordinate
        values evaluated at each configuration of the trajectory.
        The returned array has shape ``(n_frames,)``.
    """

    ################################
    # Your code goes here
    ################################

    return q

### 5C. Selecting New Shooting Points

Having defined a shooting-point selection probability, we can now use it to choose new shooting points from a reactive trajectory. The function `select_shooting_point` implements this procedure.

Given a trajectory, the first step is to evaluate the learned reaction coordinate along all configurations using the neural-network model. This produces a sequence of values

$$
(q_1,q_2,\ldots,q_N),
$$

where $N$ is the number of configurations stored along the trajectory. These values are then converted into a normalized selection probability through the Lorentzian distribution introduced above,

$$
p_{\mathrm{sel}}(q_i)
=
\frac{
\left[1+\left(q_i/\gamma\right)^2\right]^{-1}
}
{
\sum_j
\left[1+\left(q_j/\gamma\right)^2\right]^{-1}
}.
$$

The resulting probability distribution assigns a selection probability to every configuration of the trajectory. Configurations with values of $q$ close to zero are more likely to be selected, while configurations deep inside the stable basins receive a smaller weight.

Once the probability distribution has been constructed, the function performs a random draw using `np.random.choice`. A configuration index is selected according to the probabilities $p_{\mathrm{sel}}$, and the corresponding configuration is returned as the new shooting point.

Importantly, the procedure remains stochastic. Even though configurations near the transition-state region are favored, configurations located elsewhere on the trajectory can still be selected with a finite probability. This is one of the motivations for using a Lorentzian distribution: the heavy tails ensure that the selection remains focused on the transition region without becoming completely deterministic.

In a full AIMMD simulation, the selected configuration would subsequently be used to generate a new shooting move and produce additional trajectory data. In the present workshop, we instead use the selected configurations to visualize how the learned committor reshapes the shooting-point distribution and progressively concentrates sampling around the transition-state ensemble.


In [ ]:
def select_shooting_point(
    trajectory,
    model,
    selection_probability,
):
    """
    Select a shooting point from a trajectory according to a
    user-specified selection probability.

    Parameters
    ----------
    trajectory : np.ndarray
        Trajectory from which the shooting point will be selected.

    model : nn.Module
        Trained committor model used to evaluate the reaction
        coordinate q along the trajectory.

    selection_probability : callable
        Function accepting an array of reaction-coordinate values
        q and returning a normalized probability distribution of
        the same shape.

    Returns
    -------
    np.ndarray
        Selected shooting-point configuration.
    """

    ################################
    # Your code goes here
    ################################

    return shooting_point

### 5D. Selecting Shooting Points from Reactive Trajectories

We are now ready to use the learned committor to guide the selection of new shooting points. The first step is to identify the reactive trajectories contained in the initial dataset. Recall that each shooting point generated two independent trajectory segments, leading to one of three possible outcomes:

$$
(n_A,n_B)\in\{(2,0),(1,1),(0,2)\}.
$$

Only trajectories with

$$
(n_A,n_B)=(1,1)
$$

are retained in the present analysis. These trajectories contain one branch reaching state A and one branch reaching state B and therefore pass through the transition region separating the two metastable basins.

This restriction is not merely a convenience but reflects how shooting-point selection is performed in a genuine TPS or AIMMD simulation. In TPS, the path ensemble consists exclusively of reactive trajectories connecting the two states. New shooting moves are generated by selecting a configuration from one of these accepted transition paths and perturbing it. Consequently, the only configurations that can serve as shooting points are those belonging to reactive trajectories.

Nonreactive trajectories still play an important role during the simulation, but not as sources of shooting points. Rejected shooting attempts contribute to the statistical weight associated with the accepted transition paths, since a transition path that remains accepted through many consecutive shooting attempts represents a larger portion of the path ensemble. In AIMMD, the outcomes of both accepted and rejected shootings are also valuable for training the committor model because they provide information about which state is reached from a given configuration. However, these trajectories are not used directly for shooting-point selection.

There is also a physical motivation for excluding nonreactive trajectories from the selection procedure. Trajectories with outcomes $(2,0)$ or $(0,2)$ are highly likely to remain entirely on one side of the barrier and never visit the transition region. Their configurations are therefore strongly committed to one of the stable states and contain little information about the dynamical bottleneck controlling the transition. Applying the Lorentzian selection probability to such trajectories would simply concentrate probability near the least committed configurations available within a single basin, rather than near the true transition-state ensemble. By restricting the selection to reactive trajectories, we ensure that the learned committor is used in the context for which it was designed: identifying the most informative configurations along paths that actually connect states A and B.

For each reactive trajectory, the learned reaction coordinate $q$ is evaluated at every stored configuration. The Lorentzian selection probability introduced previously is then used to randomly choose a new shooting point from the trajectory. This procedure is repeated independently for every reactive trajectory.

To illustrate the effect of the width parameter $\gamma$, the selection is performed for several values of $\gamma$. Large values produce a relatively broad selection probability and therefore choose configurations from a wide portion of the reactive trajectory. As $\gamma$ decreases, the selection becomes increasingly concentrated around configurations satisfying

$$
q \approx 0 \quad \text{or, analogously,} \quad
p_B \approx \frac{1}{2}.
$$

The resulting sets of shooting points provide a direct visualization of how the learned committor can be used to focus sampling near the transition-state ensemble. In the next figure, we will compare the distributions obtained for different values of $\gamma$ and observe how the shooting points progressively collapse onto the transition-state region as the selection probability becomes more localized.

### Implement Selection of New Shooting Points from Reactive Trajectories

<div class="alert alert-block alert-info">

**TASK**

1. Identify the reactive trajectories in the dataset, i.e. the trajectories for which one branch reaches state A and the other branch reaches state B.

2. Extract the corresponding trajectories and store them in a list called `transition_paths`.

3. For each value of the Lorentzian width parameter

   $$
   \gamma \in \{1.0,;0.5,;0.1\},
   $$

   create a selection-probability function using `functools.partial`.

4. For every reactive trajectory, select a new shooting point using the learned reaction coordinate and the corresponding Lorentzian selection probability.

5. Store the resulting shooting points in a dictionary named `shooting_points_by_gamma`, using the value of $\gamma$ as the key.


</div>


In [ ]:
from functools import partial

# Select only reactive trajectories, i.e. trajectories for which
# one branch reaches A and the other reaches B.
transition_indices = ... # Your code goes here

# Extract the corresponding full trajectories.
transition_paths = ... # Your code goes here

# Values of the Lorentzian width parameter to investigate.
# Smaller values concentrate the shooting-point selection
# more strongly around q = 0.
gamma_values = [1.0, 0.5, 0.1]

# Dictionary that will store the selected shooting points
# for each value of gamma.
shooting_points_by_gamma = {}

for gamma in gamma_values:

    # Create a callable selection probability with the
    # current value of gamma fixed.
    selection_probability = partial(
        lorentzian_selection_probability,
        gamma=gamma
    )

    # Select one shooting point from each reactive trajectory
    # according to the Lorentzian selection probability.
    shooting_points_by_gamma[gamma] = ... # Your code goes here

### 5E. Visualizing the Adaptive Shooting-Point Selection

The figure below illustrates how the learned committor can be used to bias the selection of future shooting points toward the transition-state region. The two panels highlight complementary aspects of the AIMMD selection strategy.

The left panel shows the shooting points selected from the reactive trajectories for several values of the Lorentzian width parameter $\gamma$. Each point corresponds to a configuration chosen according to the probability distribution

$$
p_{\mathrm{sel}}(q)
\propto
\frac{1}{1+\left(q/\gamma\right)^2}.
$$

The effect of $\gamma$ is immediately apparent. For large values of $\gamma$, the Lorentzian is broad and the selection probability varies only weakly along the trajectory. As a result, shooting points remain relatively dispersed throughout the transition paths. As $\gamma$ decreases, the distribution becomes increasingly concentrated around configurations satisfying

$$
q \approx 0,
$$

and the selected shooting points progressively collapse onto the transition-state region. In the limit of very small $\gamma$, almost all selected configurations lie close to the learned $p_B=0.5$ contour.

The right panel explains the origin of this behavior using a single representative reactive trajectory. The trajectory is shown on top of the potential-energy surface, while the color of each configuration represents its Lorentzian weight. Bright regions correspond to configurations with a high probability of being selected, whereas dark regions correspond to configurations with a low probability.

Because the reaction coordinate changes continuously along the trajectory, the Lorentzian weights naturally peak when the trajectory passes through the vicinity of the transition state. Configurations deep inside either basin have large values of $|q|$ and therefore receive only a small statistical weight. The resulting distribution demonstrates how the learned committor acts as a filter that identifies the most informative configurations along a transition path.

This visualization captures the key idea underlying AIMMD. Rather than sampling shooting points uniformly along a trajectory, the algorithm uses the learned committor to focus computational effort on the configurations that are most relevant for describing the transition mechanism. In a full AIMMD simulation, the selected configurations would be used to generate new shooting trajectories, and the resulting data would be incorporated into a new round of committor training. Here we stop after the first iteration, but the figure clearly illustrates how the adaptive selection mechanism would progressively concentrate the sampling around the transition-state ensemble.


In [ ]:
# --------------------------------------------------
# Utility function
# --------------------------------------------------

def draw_background(ax):

    levels = np.linspace(0, 20, 21)

    ax.contour(
        X,
        Y,
        V,
        levels=levels,
        colors="0.5",
        linewidths=1
    )

    ax.contourf(
        X,
        Y,
        V,
        levels=levels,
        cmap="viridis",
        extend="max"
    )

    for label, state in zip(
        ["A", "B"],
        [state_A, state_B]
    ):

        circle = Circle(
            state.center,
            radius=state.radius,
            facecolor=(1, 1, 1, 0.5),
            edgecolor=(0, 0, 0, 1.0),
            linewidth=2.5,
            zorder=20,
        )

        ax.add_patch(circle)

        ax.text(
            state.center[0],
            state.center[1],
            label,
            ha="center",
            va="center",
            fontsize=14,
            fontweight="bold",
            color="black",
            zorder=21,
        )

    ax.set_xlim(-2.2, 2.2)
    ax.set_ylim(-2.2, 2.2)
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")


# --------------------------------------------------
# Figure
# --------------------------------------------------

fig, (ax1, ax2) = plt.subplots(
    1,
    2,
    figsize=(14, 6),
    constrained_layout=True
)

draw_background(ax1)
draw_background(ax2)

# --------------------------------------------------
# Left panel:
# selected shooting points
# --------------------------------------------------

colors = [
    "tab:blue",
    "tab:orange",
    "tab:red"
]

for gamma, color in zip(
    gamma_values,
    colors
):

    sp = shooting_points_by_gamma[gamma]

    ax1.scatter(
        sp[:, 0],
        sp[:, 1],
        s=25,
        c=color,
        alpha=0.8,
        label=rf"$\gamma={gamma}$",
        zorder=40
    )

ax1.set_title(
    "Selected shooting points"
)

ax1.legend()

# --------------------------------------------------
# Right panel:
# one trajectory + weights
# --------------------------------------------------

traj = transition_paths[0]

q = evaluate_q(
    model,
    traj
)

gamma_demo = 0.5

weights = (
    1.0 /
    (1.0 + (q / gamma_demo) ** 2)
)

ax2.plot(
    traj[:, 0],
    traj[:, 1],
    color="black",
    lw=1,
    alpha=0.4,
    zorder=25
)

sc = ax2.scatter(
    traj[:, 0],
    traj[:, 1],
    c=weights,
    cmap="plasma",
    s=25,
    zorder=30
)

cbar = plt.colorbar(
    sc,
    ax=ax2
)

cbar.set_label(
    rf"Lorentzian weight ($\gamma={gamma_demo}$)"
)

ax2.set_title(
    "Selection weights along a reactive trajectory"
)

plt.show()

### Final Discussion: From the Workshop Implementation to AIMMD

<div class="alert alert-block alert-success">

**OPTIONAL TASK (Useful to prepare the exam)**

In this workshop, we implemented a simplified version of the AIMMD algorithm introduced in [Jung2023]. Discuss the following points:

1. **Trajectory generation**

   Explain how trajectories were generated in this workshop and compare this procedure with a genuine Transition Path Sampling (TPS) simulation. In particular, discuss:

   * How shooting points were selected in our implementation.
   * How shooting points are selected in TPS.
   * The role of shooting moves and path acceptance/rejection.
   * Why trajectories generated by TPS are generally correlated, whereas the trajectories generated here are independent.

2. **Sampling of the path ensemble**

   Discuss why the trajectories produced in this notebook do not correspond to a properly sampled transition-path ensemble. What ingredients would be required to obtain the correct ensemble of reactive trajectories?

3. **Adaptive learning cycle**

   In AIMMD, the learned committor is used to improve future sampling. Describe the adaptive loop implemented in the original algorithm:

   * Generation of new shooting moves.
   * Training of the committor model.
   * Construction of the shooting-point selection probability.
   * Iterative refinement of the path ensemble.

   Explain why, in this workshop, only a single learning iteration was performed.

4. **Scaling to realistic molecular systems**

   The Wolfe–Quapp potential is a two-dimensional model for which the configuration space can be visualized directly. Discuss the challenges that would arise when studying a realistic molecular system, such as a protein folding in explicit solvent.

   Consider the following questions:

   * Why is it no longer possible to visualize the configuration space directly?
   * How does the dimensionality of the input increase?
   * What information should be provided to the neural network?
   * Why is the committor particularly valuable in high-dimensional systems?

5. **Machine-learning considerations**

   The network used in this workshop takes only two Cartesian coordinates as input. Discuss how the architecture might need to change for realistic molecular systems.

   Examples of possible considerations include:

   * Larger fully connected networks.
   * Feature engineering and collective variables.
   * Graph neural networks.
   * Equivariant neural networks.
   * Computational cost of training and inference.

</div>


## References

[Jung2023]
H. Jung, R. Covino, A. Arjun, C. Leitold, C. Dellago, P. G. Bolhuis, and G. Hummer,  
*Machine-guided path sampling to discover mechanisms of molecular self-organization*,  
Nature Computational Science **3**(4), 334–345 (2023).  
DOI: 10.1038/s43588-023-00428-z.  